# 500k CNN evaluation: M00 vs M08a vs M08

This notebook compares three controlled experiments on the same 500k BBH dataset:

- **M00** — original `SimpleCNN_Baseline`.
- **M08a** — residual encoder with ordinary convolutions, `dilations=[1,1,1]`.
- **M08** — residual encoder with multiscale dilations, `dilations=[1,2,4]`.

The comparison separates two effects:

1. **Residual encoder effect:** M08a relative to M00.
2. **Dilation effect:** M08 relative to M08a.

All event-level comparisons are aligned using the physical HDF5 indices saved in each prediction file.

## 1. Imports and configuration

Adjust `DATA_ROOT` only if the local synchronization directory differs.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

DATA_ROOT = Path("/data/vserrano/cbc_pe_data")
RESULTS_ROOT = DATA_ROOT / "results"
CHECKPOINT_ROOT = DATA_ROOT / "models" / "checkpoints"
PROCESSED_ROOT = DATA_ROOT / "processed"

DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"
LABEL_NAMES = ["chirp_mass", "total_mass", "chi_eff"]

LABEL_UNITS = {
    "chirp_mass": r"$M_\odot$",
    "total_mass": r"$M_\odot$",
    "chi_eff": "dimensionless",
}

SPLITS = ["val", "cal", "test"]
RNG_SEED = 123

print("DATA_ROOT:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())

## 2. Locate prediction, history, and checkpoint files

The resolver uses filename patterns so the notebook does not depend on manually copying very long filenames.  
If more than one candidate is found, it selects the most recently modified file and prints all candidates.

In [ ]:
def resolve_file(root: Path, patterns, description: str) -> Path:
    """
    Resolve one file from one or more glob patterns.

    Parameters
    ----------
    root
        Directory in which the patterns are evaluated.

    patterns
        A string, Path, or iterable of strings/Paths.

    description
        Human-readable description used in error messages.
    """
    if isinstance(patterns, (str, Path)):
        patterns = [patterns]

    matches = []

    for pattern in patterns:
        matches.extend(root.glob(str(pattern)))

    matches = sorted(
        set(matches),
        key=lambda path: path.stat().st_mtime,
    )

    if not matches:
        raise FileNotFoundError(
            f"Could not find {description} in {root}.\n"
            f"Patterns tried: {[str(pattern) for pattern in patterns]}"
        )

    print(f"\n{description}:")
    for path in matches:
        print("  ", path)

    selected = matches[-1]
    print("Selected:", selected)

    return selected


# FOLDERS WITH RESULTS

DATASET_RESULTS_ROOT = RESULTS_ROOT / DATASET_ID
DATASET_CHECKPOINT_ROOT = CHECKPOINT_ROOT / DATASET_ID

print("Dataset results root:", DATASET_RESULTS_ROOT)
print("Exists:", DATASET_RESULTS_ROOT.exists())

print("Dataset checkpoint root:", DATASET_CHECKPOINT_ROOT)
print("Exists:", DATASET_CHECKPOINT_ROOT.exists())



RUNS = {
    "M00": {
        "label": "M00 baseline",

        "prediction_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_Baseline_"
                "batchslices_bs256_"
                "MSELoss_seed123_"
                "val_cal_test_predictions_embeddings.npz"
            ),
        ],

        "history_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_Baseline_"
                "batchslices_bs256_"
                "MSELoss_seed123_history.npz"
            ),
        ],

        "checkpoint_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_Baseline_"
                "batchslices_bs256_"
                "MSELoss_seed123_checkpoint.pt"
            ),
        ],
    },

    "M08a": {
        "label": "M08a residual d111",

        "prediction_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_ResidualDilated_"
                "M08a_residual_emb64_d111_"
                "MSELoss_seed123_"
                "val_cal_test_predictions_embeddings.npz"
            ),
        ],

        "history_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_ResidualDilated_"
                "M08a_residual_emb64_d111_"
                "MSELoss_seed123_history.npz"
            ),
        ],

        "checkpoint_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_ResidualDilated_"
                "M08a_residual_emb64_d111_"
                "MSELoss_seed123_checkpoint.pt"
            ),
        ],
    },

    "M08": {
        "label": "M08 residual dilated d124",

        "prediction_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_ResidualDilated_"
                "batchslices_bs256_"
                "MSELoss_seed123_"
                "val_cal_test_predictions_embeddings.npz"
            ),
        ],

        "history_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_ResidualDilated_"
                "batchslices_bs256_"
                "MSELoss_seed123_history.npz"
            ),
        ],

        "checkpoint_patterns": [
            (
                f"{DATASET_ID}_"
                "SimpleCNN_ResidualDilated_"
                "batchslices_bs256_"
                "MSELoss_seed123_checkpoint.pt"
            ),
        ],
    },
}

for run_id, run in RUNS.items():
    run["prediction_path"] = resolve_file(
        DATASET_RESULTS_ROOT,
        run["prediction_patterns"],
        f"{run_id} prediction file",
    )

    run["history_path"] = resolve_file(
        DATASET_RESULTS_ROOT,
        run["history_patterns"],
        f"{run_id} history file",
    )

    run["checkpoint_path"] = resolve_file(
        DATASET_CHECKPOINT_ROOT,
        run["checkpoint_patterns"],
        f"{run_id} checkpoint",
    )

### Guard against an ambiguous M08 match

Because M08a also contains `SimpleCNN_ResidualDilated` in its name, verify that the selected M08 file does **not** contain `M08a`.

In [ ]:
assert "M08a" not in RUNS["M08"]["prediction_path"].name
assert "M08a" not in RUNS["M08"]["history_path"].name
assert "M08a" not in RUNS["M08"]["checkpoint_path"].name

for run_id, run in RUNS.items():
    print(run_id)
    print("  predictions:", run["prediction_path"].name)
    print("  history:    ", run["history_path"].name)
    print("  checkpoint: ", run["checkpoint_path"].name)

## 3. Load label statistics and prediction files

In [ ]:
LABEL_STATS_PATH = resolve_file(
    (PROCESSED_ROOT / DATASET_ID),
    [
        f"{DATASET_ID}_label_stats_train_only_train400000_val40000_cal30000_test30000_seed123.npz",
        f"{DATASET_ID}*label_stats*seed123.npz",
    ],
    "label statistics",
)

with np.load(LABEL_STATS_PATH) as stats:
    print("Label-stat keys:", stats.files)
    y_mean = np.asarray(stats["y_mean"], dtype=np.float64)
    y_std = np.asarray(stats["y_std"], dtype=np.float64)

print("y_mean:", y_mean)
print("y_std:", y_std)


def load_prediction_file(path: Path):
    out = {}
    with np.load(path, allow_pickle=True) as data:
        print(f"\n{path.name}")
        for key in sorted(data.files):
            arr = np.asarray(data[key])
            print(f"  {key:24s} shape={str(arr.shape):16s} dtype={arr.dtype}")
            out[key] = arr
    return out


RAW = {
    run_id: load_prediction_file(run["prediction_path"])
    for run_id, run in RUNS.items()
}

## 4. Align all models by physical HDF5 index

The loaders may store samples in different orders. Every comparison below uses the intersection of saved indices and reorders all arrays to a common sorted index.

In [ ]:
def align_runs_for_split(raw_by_run, split, reference_run="M00"):
    required = [f"idx_{split}", f"pred_{split}", f"y_{split}", f"emb_{split}"]

    for run_id, data in raw_by_run.items():
        missing = [key for key in required if key not in data]
        if missing:
            raise KeyError(f"{run_id} is missing keys for {split}: {missing}")

    common_idx = None
    for data in raw_by_run.values():
        idx = np.asarray(data[f"idx_{split}"], dtype=np.int64)
        common_idx = idx if common_idx is None else np.intersect1d(common_idx, idx)

    common_idx = np.sort(common_idx)
    aligned = {}

    for run_id, data in raw_by_run.items():
        idx = np.asarray(data[f"idx_{split}"], dtype=np.int64)
        order = np.argsort(idx)
        idx_sorted = idx[order]

        positions = np.searchsorted(idx_sorted, common_idx)
        if not np.array_equal(idx_sorted[positions], common_idx):
            raise RuntimeError(f"Alignment failure for {run_id}, split={split}")

        aligned[run_id] = {
            "idx": common_idx,
            "pred": np.asarray(data[f"pred_{split}"])[order][positions],
            "y": np.asarray(data[f"y_{split}"])[order][positions],
            "emb": np.asarray(data[f"emb_{split}"])[order][positions],
        }

    y_ref = aligned[reference_run]["y"]
    for run_id in aligned:
        if not np.allclose(aligned[run_id]["y"], y_ref, rtol=1e-5, atol=1e-5):
            raise AssertionError(
                f"Targets do not agree after alignment: {reference_run} vs {run_id}"
            )

    return common_idx, aligned


ALIGNED = {}
for split in SPLITS:
    common_idx, aligned = align_runs_for_split(RAW, split)
    ALIGNED[split] = aligned
    print(f"{split:5s}: {len(common_idx):,} common samples")

for split in SPLITS:
    for run_id in RUNS:
        assert ALIGNED[split][run_id]["pred"].shape[1] == 3
        assert ALIGNED[split][run_id]["emb"].shape[1] == 64

## 5. Core regression metrics

Metrics are computed in:

- **standardized space**, matching the training objective;
- **physical space**, for scientific interpretation.

In [ ]:
def inverse_standardize(y):
    return y * y_std + y_mean


def regression_metrics(y_true, y_pred, label_names, split, run_id, space):
    residual = y_pred - y_true
    abs_error = np.abs(residual)

    mse = np.mean(residual**2, axis=0)
    rmse = np.sqrt(mse)
    mae = np.mean(abs_error, axis=0)
    bias = np.mean(residual, axis=0)
    median_ae = np.median(abs_error, axis=0)
    residual_std = np.std(residual, axis=0)

    ss_res = np.sum(residual**2, axis=0)
    ss_tot = np.sum(
        (y_true - np.mean(y_true, axis=0, keepdims=True)) ** 2,
        axis=0,
    )
    r2 = 1.0 - ss_res / ss_tot

    rows = []
    for j, label in enumerate(label_names):
        rows.append({
            "run_id": run_id,
            "split": split,
            "space": space,
            "label": label,
            "MSE": float(mse[j]),
            "RMSE": float(rmse[j]),
            "MAE": float(mae[j]),
            "bias": float(bias[j]),
            "median_abs_error": float(median_ae[j]),
            "residual_std": float(residual_std[j]),
            "R2": float(r2[j]),
        })
    return rows


metric_rows = []

for split in SPLITS:
    y_std_true = ALIGNED[split]["M00"]["y"]
    y_phys_true = inverse_standardize(y_std_true)

    for run_id in RUNS:
        pred_std = ALIGNED[split][run_id]["pred"]
        pred_phys = inverse_standardize(pred_std)

        metric_rows.extend(
            regression_metrics(
                y_std_true, pred_std, LABEL_NAMES, split, run_id, "standardized"
            )
        )
        metric_rows.extend(
            regression_metrics(
                y_phys_true, pred_phys, LABEL_NAMES, split, run_id, "physical"
            )
        )

metrics_df = pd.DataFrame(metric_rows)

global_metrics_df = (
    metrics_df
    .groupby(["run_id", "split", "space"], as_index=False)
    .agg(
        global_MSE=("MSE", "mean"),
        global_RMSE=("RMSE", "mean"),
        global_MAE=("MAE", "mean"),
        mean_R2=("R2", "mean"),
    )
)

display(global_metrics_df.query("space == 'standardized'"))
display(
    metrics_df.query("split == 'test' and space == 'standardized'")
    .sort_values(["label", "run_id"])
)

## 6. Controlled deltas

For error metrics, a negative relative change means the model in the numerator improved.

- **Residual redesign:** M08a relative to M00.
- **Dilation contribution:** M08 relative to M08a.
- **Total M08 improvement:** M08 relative to M00.

In [ ]:
def build_delta_table(metrics, split="test", space="standardized"):
    sub = metrics.query("split == @split and space == @space").copy()

    piv = sub.pivot(
        index="label",
        columns="run_id",
        values=["MSE", "RMSE", "MAE", "R2", "bias"],
    )

    rows = []
    comparisons = [
        ("M08a_vs_M00", "M00", "M08a"),
        ("M08_vs_M08a", "M08a", "M08"),
        ("M08_vs_M00", "M00", "M08"),
    ]

    for comparison, old, new in comparisons:
        for label in LABEL_NAMES:
            row = {
                "comparison": comparison,
                "label": label,
            }

            for metric in ["MSE", "RMSE", "MAE"]:
                old_value = float(piv.loc[label, (metric, old)])
                new_value = float(piv.loc[label, (metric, new)])
                row[f"{metric}_old"] = old_value
                row[f"{metric}_new"] = new_value
                row[f"{metric}_relative_change_pct"] = (
                    100.0 * (new_value - old_value) / old_value
                )

            row["R2_old"] = float(piv.loc[label, ("R2", old)])
            row["R2_new"] = float(piv.loc[label, ("R2", new)])
            row["R2_delta"] = row["R2_new"] - row["R2_old"]

            row["bias_old"] = float(piv.loc[label, ("bias", old)])
            row["bias_new"] = float(piv.loc[label, ("bias", new)])
            row["abs_bias_delta"] = abs(row["bias_new"]) - abs(row["bias_old"])

            rows.append(row)

    return pd.DataFrame(rows)


delta_test_std_df = build_delta_table(
    metrics_df, split="test", space="standardized"
)

display(delta_test_std_df)

global_test_std = (
    global_metrics_df
    .query("split == 'test' and space == 'standardized'")
    .set_index("run_id")
)

global_delta_rows = []
for comparison, old, new in [
    ("M08a_vs_M00", "M00", "M08a"),
    ("M08_vs_M08a", "M08a", "M08"),
    ("M08_vs_M00", "M00", "M08"),
]:
    global_delta_rows.append({
        "comparison": comparison,
        "global_MSE_old": global_test_std.loc[old, "global_MSE"],
        "global_MSE_new": global_test_std.loc[new, "global_MSE"],
        "global_MSE_relative_change_pct": 100 * (
            global_test_std.loc[new, "global_MSE"]
            - global_test_std.loc[old, "global_MSE"]
        ) / global_test_std.loc[old, "global_MSE"],
        "global_MAE_relative_change_pct": 100 * (
            global_test_std.loc[new, "global_MAE"]
            - global_test_std.loc[old, "global_MAE"]
        ) / global_test_std.loc[old, "global_MAE"],
    })

global_delta_df = pd.DataFrame(global_delta_rows)
display(global_delta_df)

## 7. Paired bootstrap uncertainty

Because all models predict the same test events, the correct uncertainty analysis is paired.

For each event and target:

$$
\Delta_i = \ell_{\mathrm{new},i} - \ell_{\mathrm{old},i}
$$

A 95% confidence interval fully below zero supports an improvement by the new model.

In [ ]:
def paired_bootstrap_delta(
    y_true,
    pred_old,
    pred_new,
    metric="squared_error",
    n_boot=3000,
    seed=123,
):
    if metric == "squared_error":
        loss_old = (pred_old - y_true) ** 2
        loss_new = (pred_new - y_true) ** 2
    elif metric == "absolute_error":
        loss_old = np.abs(pred_old - y_true)
        loss_new = np.abs(pred_new - y_true)
    else:
        raise ValueError(f"Unsupported metric: {metric}")

    delta = loss_new - loss_old
    observed = np.mean(delta, axis=0)

    rng = np.random.default_rng(seed)
    n = len(y_true)
    boot = np.empty((n_boot, y_true.shape[1]), dtype=np.float64)

    # Sample paired rows. Chunked loop keeps memory use moderate.
    for b in range(n_boot):
        sample_idx = rng.integers(0, n, size=n)
        boot[b] = np.mean(delta[sample_idx], axis=0)

    lower = np.quantile(boot, 0.025, axis=0)
    upper = np.quantile(boot, 0.975, axis=0)

    return observed, lower, upper


test = ALIGNED["test"]
y_true_test = test["M00"]["y"]

bootstrap_rows = []

for comparison, old, new in [
    ("M08a_vs_M00", "M00", "M08a"),
    ("M08_vs_M08a", "M08a", "M08"),
    ("M08_vs_M00", "M00", "M08"),
]:
    for metric in ["squared_error", "absolute_error"]:
        observed, low, high = paired_bootstrap_delta(
            y_true_test,
            test[old]["pred"],
            test[new]["pred"],
            metric=metric,
            n_boot=3000,
            seed=RNG_SEED,
        )

        for j, label in enumerate(LABEL_NAMES):
            bootstrap_rows.append({
                "comparison": comparison,
                "metric": metric,
                "label": label,
                "mean_delta_new_minus_old": observed[j],
                "ci95_low": low[j],
                "ci95_high": high[j],
                "supports_improvement": high[j] < 0,
                "supports_degradation": low[j] > 0,
            })

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df.sort_values(["comparison", "metric", "label"]))

## 8. Per-event win rate

This distinguishes a broad small improvement from a gain driven by a small number of difficult events.

In [ ]:
def per_sample_win_rate(y_true, pred_old, pred_new, comparison):
    ae_old = np.abs(pred_old - y_true)
    ae_new = np.abs(pred_new - y_true)

    rows = []
    for j, label in enumerate(LABEL_NAMES):
        rows.append({
            "comparison": comparison,
            "label": label,
            "new_win_rate": float(np.mean(ae_new[:, j] < ae_old[:, j])),
            "tie_rate": float(np.mean(ae_new[:, j] == ae_old[:, j])),
            "old_win_rate": float(np.mean(ae_old[:, j] < ae_new[:, j])),
            "mean_AE_delta_new_minus_old": float(
                np.mean(ae_new[:, j] - ae_old[:, j])
            ),
        })
    return rows


win_rate_rows = []
for comparison, old, new in [
    ("M08a_vs_M00", "M00", "M08a"),
    ("M08_vs_M08a", "M08a", "M08"),
    ("M08_vs_M00", "M00", "M08"),
]:
    win_rate_rows.extend(
        per_sample_win_rate(
            y_true_test,
            test[old]["pred"],
            test[new]["pred"],
            comparison,
        )
    )

win_rate_df = pd.DataFrame(win_rate_rows)
display(win_rate_df)

## 9. Contraction and regression-to-the-mean diagnostics

The ordinary min–max range is sensitive to a few extreme predictions.  
The robust diagnostics below include:

- regression slope and intercept;
- Pearson correlation;
- `std(pred) / std(true)`;
- central 1–99% range ratio.

In [ ]:
def slope_diagnostics(y_true, y_pred, run_id, split, space):
    rows = []

    for j, label in enumerate(LABEL_NAMES):
        true = y_true[:, j]
        pred = y_pred[:, j]

        slope, intercept = np.polyfit(true, pred, deg=1)
        correlation = np.corrcoef(true, pred)[0, 1]

        true_std = np.std(true)
        pred_std = np.std(pred)

        true_q01, true_q99 = np.quantile(true, [0.01, 0.99])
        pred_q01, pred_q99 = np.quantile(pred, [0.01, 0.99])

        rows.append({
            "run_id": run_id,
            "split": split,
            "space": space,
            "label": label,
            "slope": float(slope),
            "intercept": float(intercept),
            "correlation": float(correlation),
            "std_ratio_pred_over_true": float(pred_std / true_std),
            "q01_q99_ratio_pred_over_true": float(
                (pred_q99 - pred_q01) / (true_q99 - true_q01)
            ),
        })

    return rows


slope_rows = []
for split in SPLITS:
    y_std_true = ALIGNED[split]["M00"]["y"]
    y_phys_true = inverse_standardize(y_std_true)

    for run_id in RUNS:
        pred_std = ALIGNED[split][run_id]["pred"]
        pred_phys = inverse_standardize(pred_std)

        slope_rows.extend(
            slope_diagnostics(
                y_std_true, pred_std, run_id, split, "standardized"
            )
        )
        slope_rows.extend(
            slope_diagnostics(
                y_phys_true, pred_phys, run_id, split, "physical"
            )
        )

slope_df = pd.DataFrame(slope_rows)

display(
    slope_df.query("split == 'test' and space == 'standardized'")
    .sort_values(["label", "run_id"])
)

In [ ]:
for label in LABEL_NAMES:
    sub = slope_df.query(
        "split == 'test' and space == 'standardized' and label == @label"
    ).set_index("run_id").loc[list(RUNS)]

    plt.figure(figsize=(7.5, 4.5))
    plt.plot(sub.index, sub["slope"], marker="o", label="slope")
    plt.plot(
        sub.index,
        sub["std_ratio_pred_over_true"],
        marker="s",
        label="std ratio",
    )
    plt.axhline(1.0, linewidth=1, linestyle="--")
    plt.ylabel("Ratio / slope")
    plt.title(f"Contraction diagnostics — {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 10. Absolute-error quantiles

Upper quantiles are important for difficult events and later conformal calibration.

In [ ]:
ERROR_QUANTILES = [0.50, 0.90, 0.95, 0.99]

quantile_rows = []

for split in SPLITS:
    y_phys_true = inverse_standardize(ALIGNED[split]["M00"]["y"])

    for run_id in RUNS:
        pred_phys = inverse_standardize(ALIGNED[split][run_id]["pred"])
        abs_error = np.abs(pred_phys - y_phys_true)

        for j, label in enumerate(LABEL_NAMES):
            for q in ERROR_QUANTILES:
                quantile_rows.append({
                    "run_id": run_id,
                    "split": split,
                    "space": "physical",
                    "label": label,
                    "quantile": q,
                    "absolute_error": float(np.quantile(abs_error[:, j], q)),
                })

quantiles_df = pd.DataFrame(quantile_rows)

test_quantiles = (
    quantiles_df.query("split == 'test'")
    .pivot_table(
        index=["label", "quantile"],
        columns="run_id",
        values="absolute_error",
    )
    .reset_index()
)

test_quantiles["M08a_vs_M00_pct"] = (
    100 * (test_quantiles["M08a"] - test_quantiles["M00"])
    / test_quantiles["M00"]
)
test_quantiles["M08_vs_M08a_pct"] = (
    100 * (test_quantiles["M08"] - test_quantiles["M08a"])
    / test_quantiles["M08a"]
)
test_quantiles["M08_vs_M00_pct"] = (
    100 * (test_quantiles["M08"] - test_quantiles["M00"])
    / test_quantiles["M00"]
)

display(test_quantiles)

## 11. Error by true-value quantile bin

The same bin edges are used for every model.  
The key dilation plot is `M08 − M08a`: values below zero indicate improvement from dilation.

In [ ]:
def make_unique_quantile_edges(values, n_bins):
    edges = np.quantile(values, np.linspace(0, 1, n_bins + 1))
    edges = np.unique(edges)

    if len(edges) < n_bins + 1:
        warnings.warn(
            f"Repeated quantile edges: requested {n_bins} bins, "
            f"obtained {len(edges) - 1}."
        )
    return edges


def binned_error_metrics(
    y_true,
    predictions_by_run,
    n_bins=8,
    split="test",
    space="physical",
):
    rows = []

    for j, label in enumerate(LABEL_NAMES):
        true = y_true[:, j]
        edges = make_unique_quantile_edges(true, n_bins)

        # Include the rightmost edge.
        bin_id = np.digitize(true, edges[1:-1], right=False)

        for b in range(len(edges) - 1):
            mask = bin_id == b
            count = int(mask.sum())

            if count == 0:
                continue

            true_bin = true[mask]

            for run_id, pred_all in predictions_by_run.items():
                pred = pred_all[mask, j]
                residual = pred - true_bin
                ae = np.abs(residual)

                rows.append({
                    "split": split,
                    "space": space,
                    "label": label,
                    "bin": b,
                    "low": float(edges[b]),
                    "high": float(edges[b + 1]),
                    "true_mean": float(np.mean(true_bin)),
                    "count": count,
                    "run_id": run_id,
                    "bias_pred_minus_true": float(np.mean(residual)),
                    "abs_bias": float(abs(np.mean(residual))),
                    "MAE": float(np.mean(ae)),
                    "RMSE": float(np.sqrt(np.mean(residual**2))),
                    "q90_abs_error": float(np.quantile(ae, 0.90)),
                    "q95_abs_error": float(np.quantile(ae, 0.95)),
                })

    return pd.DataFrame(rows)


test_y_phys = inverse_standardize(ALIGNED["test"]["M00"]["y"])
test_pred_phys = {
    run_id: inverse_standardize(ALIGNED["test"][run_id]["pred"])
    for run_id in RUNS
}

binned_df = binned_error_metrics(
    test_y_phys,
    test_pred_phys,
    n_bins=8,
)

display(binned_df.head())

In [ ]:
def binned_delta_frame(binned, old, new, comparison):
    key_cols = [
        "split", "space", "label", "bin",
        "low", "high", "true_mean", "count",
    ]

    old_df = (
        binned.query("run_id == @old")
        [key_cols + ["bias_pred_minus_true", "abs_bias", "MAE", "RMSE",
                     "q90_abs_error", "q95_abs_error"]]
        .rename(columns={
            "bias_pred_minus_true": "bias_old",
            "abs_bias": "abs_bias_old",
            "MAE": "MAE_old",
            "RMSE": "RMSE_old",
            "q90_abs_error": "q90_old",
            "q95_abs_error": "q95_old",
        })
    )

    new_df = (
        binned.query("run_id == @new")
        [key_cols + ["bias_pred_minus_true", "abs_bias", "MAE", "RMSE",
                     "q90_abs_error", "q95_abs_error"]]
        .rename(columns={
            "bias_pred_minus_true": "bias_new",
            "abs_bias": "abs_bias_new",
            "MAE": "MAE_new",
            "RMSE": "RMSE_new",
            "q90_abs_error": "q90_new",
            "q95_abs_error": "q95_new",
        })
    )

    merged = old_df.merge(new_df, on=key_cols, validate="one_to_one")
    merged.insert(0, "comparison", comparison)

    for metric in ["abs_bias", "MAE", "RMSE", "q90", "q95"]:
        merged[f"{metric}_delta_new_minus_old"] = (
            merged[f"{metric}_new"] - merged[f"{metric}_old"]
        )

    return merged


binned_delta_df = pd.concat([
    binned_delta_frame(binned_df, "M00", "M08a", "M08a_vs_M00"),
    binned_delta_frame(binned_df, "M08a", "M08", "M08_vs_M08a"),
    binned_delta_frame(binned_df, "M00", "M08", "M08_vs_M00"),
], ignore_index=True)

display(
    binned_delta_df.query("comparison == 'M08_vs_M08a'")
    .sort_values(["label", "bin"])
)

In [ ]:
for label in LABEL_NAMES:
    sub = (
        binned_delta_df
        .query("comparison == 'M08_vs_M08a' and label == @label")
        .sort_values("true_mean")
    )

    plt.figure(figsize=(7.5, 4.5))
    plt.axhline(0, linewidth=1)
    plt.plot(
        sub["true_mean"],
        sub["MAE_delta_new_minus_old"],
        marker="o",
        label="Δ MAE",
    )
    plt.plot(
        sub["true_mean"],
        sub["q90_delta_new_minus_old"],
        marker="s",
        label="Δ q90 |error|",
    )
    plt.xlabel(f"True {label} [{LABEL_UNITS[label]}]")
    plt.ylabel("M08 − M08a")
    plt.title(f"Effect of dilation by true-value bin — {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 12. Prediction scatter and residual plots

These plots are subsampled only for rendering; all numerical metrics use the complete test set.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
n_plot = min(5000, len(test_y_phys))
plot_idx = rng.choice(len(test_y_phys), size=n_plot, replace=False)

for j, label in enumerate(LABEL_NAMES):
    true = test_y_phys[plot_idx, j]

    for run_id in RUNS:
        pred = test_pred_phys[run_id][plot_idx, j]

        plt.figure(figsize=(5.5, 5.0))
        plt.scatter(true, pred, s=5, alpha=0.25)

        lo = min(true.min(), pred.min())
        hi = max(true.max(), pred.max())
        plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)

        plt.xlabel(f"True {label}")
        plt.ylabel(f"Predicted {label}")
        plt.title(f"{run_id} — {label}")
        plt.tight_layout()
        plt.show()

In [ ]:
for j, label in enumerate(LABEL_NAMES):
    plt.figure(figsize=(7.5, 4.8))

    for run_id in RUNS:
        true = test_y_phys[plot_idx, j]
        pred = test_pred_phys[run_id][plot_idx, j]
        residual = pred - true

        plt.scatter(true, residual, s=5, alpha=0.20, label=run_id)

    plt.axhline(0, linewidth=1, linestyle="--")
    plt.xlabel(f"True {label} [{LABEL_UNITS[label]}]")
    plt.ylabel("Prediction − truth")
    plt.title(f"Residual comparison — {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 13. Training-history comparison

The history loader is defensive because older runs may use slightly different key names.

In [ ]:
def load_history(path):
    with np.load(path, allow_pickle=True) as data:
        history = {key: np.asarray(data[key]) for key in data.files}
    print(path.name, "keys:", sorted(history))
    return history


HISTORIES = {
    run_id: load_history(run["history_path"])
    for run_id, run in RUNS.items()
}


def first_existing(mapping, candidates):
    for key in candidates:
        if key in mapping:
            return np.asarray(mapping[key])
    return None


history_rows = []

for run_id, history in HISTORIES.items():
    train_loss = first_existing(
        history, ["train_loss", "train_losses", "loss_train"]
    )
    val_loss = first_existing(
        history, ["val_loss", "val_losses", "loss_val"]
    )

    if train_loss is None or val_loss is None:
        print(f"Skipping history summary for {run_id}: loss keys not found.")
        continue

    best_zero_based = int(np.nanargmin(val_loss))
    best_epoch = best_zero_based + 1
    stop_epoch = len(val_loss)

    elapsed = first_existing(
        history,
        ["elapsed_seconds", "elapsed_time", "epoch_times", "total_times"],
    )

    elapsed_hours = np.nan
    if elapsed is not None:
        elapsed = np.asarray(elapsed, dtype=float)
        # If per-epoch durations, sum; if cumulative, use final value.
        if len(elapsed) == len(val_loss):
            if np.all(np.diff(elapsed) >= 0) and elapsed[-1] > elapsed[:-1].sum() * 0.5:
                elapsed_hours = float(elapsed[-1] / 3600)
            else:
                elapsed_hours = float(np.sum(elapsed) / 3600)

    history_rows.append({
        "run_id": run_id,
        "best_epoch": best_epoch,
        "stop_epoch": stop_epoch,
        "best_val_loss": float(val_loss[best_zero_based]),
        "elapsed_hours": elapsed_hours,
    })

    epochs = np.arange(1, len(train_loss) + 1)

    plt.figure(figsize=(8.0, 4.8))
    plt.plot(epochs, train_loss, label="train")
    plt.plot(epochs, val_loss, label="validation")
    plt.axvline(best_epoch, linestyle="--", linewidth=1, label="best epoch")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.title(f"Training history — {run_id}")
    plt.legend()
    plt.tight_layout()
    plt.show()

history_summary_df = pd.DataFrame(history_rows)
display(history_summary_df)

In [ ]:
plt.figure(figsize=(8.5, 5.0))

for run_id, history in HISTORIES.items():
    val_loss = first_existing(
        history, ["val_loss", "val_losses", "loss_val"]
    )
    if val_loss is None:
        continue
    epochs = np.arange(1, len(val_loss) + 1)
    plt.plot(epochs, val_loss, label=run_id)

plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.title("Validation-loss comparison")
plt.legend()
plt.tight_layout()
plt.show()

## 14. Parameter counts and checkpoint metadata

This cell requires PyTorch and the project source tree to be importable.  
If the local environment cannot load the old checkpoint, the rest of the notebook remains valid.

In [ ]:
checkpoint_rows = []

try:
    import torch

    for run_id, run in RUNS.items():
        checkpoint = torch.load(run["checkpoint_path"], map_location="cpu")

        state = checkpoint.get("model_state_dict", checkpoint.get("state_dict"))
        parameter_like_count = (
            int(sum(t.numel() for t in state.values()))
            if state is not None else np.nan
        )

        model_config = checkpoint.get("model_config", {})
        training_config = checkpoint.get("training_config", {})

        checkpoint_rows.append({
            "run_id": run_id,
            "epoch_saved": checkpoint.get("epoch", np.nan),
            "best_val_loss_checkpoint": checkpoint.get("best_val_loss", np.nan),
            "elapsed_hours_checkpoint": (
                checkpoint.get("elapsed_seconds", np.nan) / 3600
                if checkpoint.get("elapsed_seconds") is not None
                else np.nan
            ),
            "parameter_like_count_from_state": parameter_like_count,
            "class_name": model_config.get("class_name"),
            "model_kwargs": model_config.get("kwargs"),
            "batch_size": training_config.get("batch_size"),
        })

    checkpoint_df = pd.DataFrame(checkpoint_rows)
    display(checkpoint_df)

except Exception as exc:
    print("Checkpoint metadata inspection skipped:")
    print(type(exc).__name__, exc)
    checkpoint_df = pd.DataFrame()

## 15. Compact decision table

The primary scientific comparison is **M08 vs M08a**:

- error deltas quantify the practical gain from dilation;
- paired bootstrap tests whether the gain is robust across test events;
- slope and dispersion ratios indicate whether dilation reduces contraction;
- bin and tail metrics show where the gain occurs.

In [ ]:
dilation_delta = (
    delta_test_std_df
    .query("comparison == 'M08_vs_M08a'")
    [[
        "label",
        "MSE_relative_change_pct",
        "MAE_relative_change_pct",
        "R2_delta",
        "abs_bias_delta",
    ]]
)

dilation_boot_mse = (
    bootstrap_df
    .query(
        "comparison == 'M08_vs_M08a' "
        "and metric == 'squared_error'"
    )
    [[
        "label",
        "mean_delta_new_minus_old",
        "ci95_low",
        "ci95_high",
        "supports_improvement",
    ]]
    .rename(columns={
        "mean_delta_new_minus_old": "MSE_delta",
        "ci95_low": "MSE_ci95_low",
        "ci95_high": "MSE_ci95_high",
        "supports_improvement": "MSE_bootstrap_supports_improvement",
    })
)

dilation_boot_mae = (
    bootstrap_df
    .query(
        "comparison == 'M08_vs_M08a' "
        "and metric == 'absolute_error'"
    )
    [[
        "label",
        "mean_delta_new_minus_old",
        "ci95_low",
        "ci95_high",
        "supports_improvement",
    ]]
    .rename(columns={
        "mean_delta_new_minus_old": "MAE_delta",
        "ci95_low": "MAE_ci95_low",
        "ci95_high": "MAE_ci95_high",
        "supports_improvement": "MAE_bootstrap_supports_improvement",
    })
)

dilation_slopes = (
    slope_df
    .query("split == 'test' and space == 'standardized'")
    .pivot(
        index="label",
        columns="run_id",
        values=["slope", "std_ratio_pred_over_true",
                "q01_q99_ratio_pred_over_true"],
    )
)

dilation_slope_rows = []
for label in LABEL_NAMES:
    dilation_slope_rows.append({
        "label": label,
        "slope_M08a": dilation_slopes.loc[label, ("slope", "M08a")],
        "slope_M08": dilation_slopes.loc[label, ("slope", "M08")],
        "slope_delta": (
            dilation_slopes.loc[label, ("slope", "M08")]
            - dilation_slopes.loc[label, ("slope", "M08a")]
        ),
        "std_ratio_M08a": dilation_slopes.loc[
            label, ("std_ratio_pred_over_true", "M08a")
        ],
        "std_ratio_M08": dilation_slopes.loc[
            label, ("std_ratio_pred_over_true", "M08")
        ],
    })

dilation_slope_df = pd.DataFrame(dilation_slope_rows)

decision_df = (
    dilation_delta
    .merge(dilation_boot_mse, on="label")
    .merge(dilation_boot_mae, on="label")
    .merge(dilation_slope_df, on="label")
)

display(decision_df)

## 16. Save tables

The notebook saves all derived tables to a dedicated comparison directory.

In [ ]:
OUTPUT_DIR = RESULTS_ROOT / "evaluation_M00_M08a_M08"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "metrics_all.csv": metrics_df,
    "global_metrics.csv": global_metrics_df,
    "test_deltas.csv": delta_test_std_df,
    "global_test_deltas.csv": global_delta_df,
    "paired_bootstrap.csv": bootstrap_df,
    "win_rates.csv": win_rate_df,
    "slope_diagnostics.csv": slope_df,
    "absolute_error_quantiles.csv": quantiles_df,
    "binned_metrics.csv": binned_df,
    "binned_deltas.csv": binned_delta_df,
    "training_summary.csv": history_summary_df,
    "dilation_decision_table.csv": decision_df,
}

if not checkpoint_df.empty:
    tables["checkpoint_metadata.csv"] = checkpoint_df

for filename, dataframe in tables.items():
    path = OUTPUT_DIR / filename
    dataframe.to_csv(path, index=False)
    print("Saved:", path)

## 17. Decision

Complete this section after executing all cells.

### Residual-encoder effect: M08a vs M00

- Global test MSE change:
- Per-target changes:
- Bootstrap support:
- Tail and bin behavior:

### Dilation effect: M08 vs M08a

- Global test MSE change:
- Per-target changes:
- Paired-bootstrap support:
- Change in `chi_eff` slope:
- Change in tail errors:
- Regions where dilation helps or hurts:

### Recommended model

State whether:

1. M08 remains the operational baseline;
2. dilation is supported as a useful component;
3. a second training seed is required before the next architectural experiment;
4. a parallel multiscale architecture is justified.

## CONCLUSION

### Residual-encoder effect: M08a vs M00

The residual non-dilated architecture M08a substantially improves the original M00 baseline. On the standardized test set, M08a reduces the global MSE by 12.96% and the global MAE by 7.39%. The MSE decreases consistently for all three targets: 13.06% for chirp mass, 13.34% for total mass, and 12.76% for effective spin.

The paired bootstrap confidence intervals for both squared and absolute error are fully below zero for all targets, supporting a robust improvement over the same test events. Therefore, the main performance gain of the M08 architecture is not caused solely by dilation. The deeper residual encoder, reduced temporal downsampling, and skip connections already provide a substantially better signal representation than M00.

### Dilation effect: M08 vs M08a

Keeping the architecture, number of parameters, training configuration, and random seed fixed, replacing dilations `[1,1,1]` with `[1,2,4]` produces an additional 3.84% reduction in global test MSE and a 1.24% reduction in global MAE.

The target-wise MSE improvements are 2.64% for chirp mass, 3.05% for total mass, and 4.74% for effective spin. All paired-bootstrap confidence intervals for MSE and MAE remain below zero, indicating that the improvement is statistically supported across the test events.

The effect is strongest for effective spin. Its regression slope increases from 0.790 to 0.812, while the predicted-to-true standard-deviation ratio increases from 0.879 to 0.898. The upper error tail also improves progressively, with reductions of 2.74%, 3.00%, and 5.23% at the q90, q95, and q99 absolute-error quantiles. This suggests that the enlarged multiscale temporal context is especially useful for difficult effective-spin events.

For the mass parameters, M08 reduces MSE, MAE, and upper-tail errors, although the regression slopes and prediction ranges become slightly more contracted. Thus, dilation improves overall accuracy but does not remove regression-to-the-mean effects for the mass targets.

### Recommended model

M08 should replace M00 as the operational baseline. The results support both parts of the architectural redesign:

1. The residual encoder accounts for most of the improvement over M00.
2. The increasing dilation pattern `[1,2,4]` provides an additional, independently measurable gain over the equivalent non-dilated residual architecture.

M08 reaches its best validation checkpoint at epoch 36, compared with epoch 57 for M08a and epoch 135 for M00. It also achieves the best test performance while using the same number of parameters as M08a.

A second training seed should be run before making a strong claim about optimization-level reproducibility. However, the paired event-level analysis already provides clear evidence that dilation improves the current seed. The next architecture should therefore retain the residual dilated encoder.
